# Task 7: Refine Consensus Labels and Gather Evidence for Cluster Labels

This notebook performs cluster refinement, subsample stability analysis, keyword extraction, centrality/boundary profiling, and structural/geographic crosstabs on the final constitutional preamble dataset:
1. **Refinement & Confidence**:
   - Rebuilding co-association matrix $P$ across 200 random KMeans initializations on $C_c$.
   - Iteratively refining consensus assignments by reassigning countries to the cluster with the highest mean co-association.
   - Assessing subsampling stability ($P_{\text{sub}}$) across 200 iterations on random 80% subsamples without replacement.
   - Identifying boundary countries where confidence is low or runner-up score exceeds internal cluster cohesion.
2. **Keywords per Final Cluster**:
   - Class-based TF-IDF (c-TF-IDF) on concatenated cluster texts for top 15 unigrams and top 10 bigrams.
   - Document contrast filtering to extract discriminative unigrams (in-cluster prevalence $\ge 30\%$, ratio $\ge 2.0$).
3. **Central & Boundary Countries**:
   - Top 5 central preambles closest to cluster centroid by cosine similarity (with 300-char excerpts).
   - Top 5 lowest subsample-confidence members and length profiles per cluster.
4. **Opening Formula**:
   - Structural taxonomy of preamble opening clauses (`we_the_people`, `in_the_name_of`, `whereas`, `other`).
   - Contingency analysis and Cramer's V association with content clusters.
5. **Geographic Distribution**:
   - Integration with Natural Earth country geometries as specified in `app.py`.
   - Continent alignment, unmatched island/microstate inventory, and Cramer's V.
6. **Data Exports**:
   - Persisting `data/processed/cluster_final.csv` and `data/processed/cluster_keywords.csv`.


## 1. Input Verification and Centered Masked Embeddings ($C_c$)

Load the pre-computed consensus results, country order, final preamble dataset, and masked embeddings. Reconstruct the centered, L2-normalized representation $C_c$ exactly as defined in Notebook 05.

In [1]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score
from sklearn.feature_extraction.text import CountVectorizer
from scipy.stats import chi2_contingency
from scipy.stats.contingency import association
import geopandas as gpd

data_dir = '../data/processed'

# 1. Load inputs
order_df = pd.read_csv(os.path.join(data_dir, 'final_country_order.csv'))
cons_df = pd.read_csv(os.path.join(data_dir, 'cluster_consensus.csv'))
preamble_df = pd.read_csv(os.path.join(data_dir, 'preamble_final.csv'))
clusterable_df = preamble_df[preamble_df['status'].isin(['preamble_ok', 'preamble_cut'])].copy().reset_index(drop=True)
emb_masked = np.load(os.path.join(data_dir, 'emb_final_masked.npy'))

# 2. Assertions
assert len(order_df) == 158, f"Expected 158 rows in order file, got {len(order_df)}"
assert (order_df['Country'] == cons_df['Country']).all(), "Mismatch between order file and consensus file countries!"
assert (order_df['Country'] == clusterable_df['Country']).all(), "Mismatch between order file and clusterable preambles!"
assert emb_masked.shape == (158, 768), f"Expected shape (158, 768), got {emb_masked.shape}"

# Check cons_C_c cluster sizes
cons_sizes = cons_df['cons_C_c'].value_counts().tolist()
assert cons_sizes == [30, 27, 25, 23, 22, 16, 15], f"Expected cons_C_c sizes [30, 27, 25, 23, 22, 16, 15], got {cons_sizes}"
print(f"Validated inputs: 158 aligned countries. cons_C_c cluster sizes: {cons_sizes}")

# 3. Construct C_c (subtract column mean, L2-normalize rows)
emb_C_c = emb_masked - emb_masked.mean(axis=0, keepdims=True)
emb_C_c = emb_C_c / np.linalg.norm(emb_C_c, axis=1, keepdims=True)
print(f"Constructed C_c with shape {emb_C_c.shape}, unit row norms verified.")


Validated inputs: 158 aligned countries. cons_C_c cluster sizes: [30, 27, 25, 23, 22, 16, 15]
Constructed C_c with shape (158, 768), unit row norms verified.


## 2. Co-association Matrix $P$ and Iterative Label Refinement

Rebuild the co-association matrix $P$ across 200 random KMeans initializations ($k=7, n_{\text{init}}=10, s \in [0, 199]$) on $C_c$.

Starting from the initial consensus labels `cons_C_c`, iteratively reassign each country to the cluster maximizing mean co-association with other members until convergence (or 20 passes).

In [2]:
# 1. Rebuild co-association matrix P on C_c
N = len(order_df)
P = np.zeros((N, N), dtype=np.float64)

for s in range(200):
    km = KMeans(n_clusters=7, n_init=10, random_state=s)
    labs = km.fit_predict(emb_C_c)
    P += (labs[:, None] == labs[None, :])
P /= 200.0

# 2. Iterative refinement from cons_C_c
current_labels = cons_df['cons_C_c'].values.copy()
print("Starting refinement from cons_C_c (sizes: {})".format(pd.Series(current_labels).value_counts().to_dict()))

all_moves = []
for p in range(20):
    new_labels = current_labels.copy()
    moves_in_pass = []
    for i in range(N):
        own_c = current_labels[i]
        scores = {}
        for c in range(7):
            members = [j for j in range(N) if current_labels[j] == c and j != i]
            scores[c] = np.mean(P[i, members]) if len(members) > 0 else 0.0
        best_c = max(scores.keys(), key=lambda c: scores[c])
        if best_c != own_c:
            moves_in_pass.append({
                'pass': p + 1,
                'country': order_df.loc[i, 'Country'],
                'from': own_c,
                'to': best_c,
                'old_mean_P': scores[own_c],
                'new_mean_P': scores[best_c]
            })
            new_labels[i] = best_c
            
    print(f"Pass {p + 1}: {len(moves_in_pass)} moves")
    for m in moves_in_pass:
        print(f"  {m['country']:<20}: {m['from']} -> {m['to']} (old mean P: {m['old_mean_P']:.4f}, new mean P: {m['new_mean_P']:.4f})")
    all_moves.extend(moves_in_pass)
    
    if np.array_equal(new_labels, current_labels):
        print(f"Converged at pass {p + 1}!")
        break
    current_labels = new_labels

# 3. Renumber final clusters from largest to smallest
refined_counts = pd.Series(current_labels).value_counts()
print(f"\nRefined cluster sizes before renumbering: {refined_counts.to_dict()}")

sorted_old_clusters = refined_counts.index.tolist()
old_to_new = {old_c: new_c for new_c, old_c in enumerate(sorted_old_clusters)}

print("\nOld-to-New Numbering Table:")
renumber_df = pd.DataFrame({
    'old_cluster': sorted_old_clusters,
    'new_cluster': [old_to_new[c] for c in sorted_old_clusters],
    'size': [refined_counts[c] for c in sorted_old_clusters]
})
print(renumber_df.to_string(index=False))

final_cluster = np.array([old_to_new[c] for c in current_labels])
final_sizes = pd.Series(final_cluster).value_counts().tolist()
print(f"\nFinal cluster sizes (final_cluster): {final_sizes}")


Starting refinement from cons_C_c (sizes: {0: 30, 1: 27, 2: 25, 3: 23, 4: 22, 5: 16, 6: 15})
Pass 1: 3 moves
  Algeria             : 1 -> 5 (old mean P: 0.3177, new mean P: 0.6597)
  Ethiopia            : 4 -> 0 (old mean P: 0.2579, new mean P: 0.3062)
  Marshall Islands    : 0 -> 4 (old mean P: 0.3950, new mean P: 0.4661)
Pass 2: 0 moves
Converged at pass 2!

Refined cluster sizes before renumbering: {0: 30, 1: 26, 2: 25, 3: 23, 4: 22, 5: 17, 6: 15}

Old-to-New Numbering Table:
 old_cluster  new_cluster  size
           0            0    30
           1            1    26
           2            2    25
           3            3    23
           4            4    22
           5            5    17
           6            6    15

Final cluster sizes (final_cluster): [30, 26, 25, 23, 22, 17, 15]


## 3. Subsampling Stability ($P_{\text{sub}}$), Consensus Verification, and Boundary Flags

Evaluate stability under $80\%$ subsampling (126 of 158 countries drawn without replacement) across 200 iterations:
$$P_{\text{sub}}[i, j] = \frac{\text{runs where } i, j \text{ are both drawn and share a cluster}}{\text{runs where } i, j \text{ are both drawn}}$$

Compute:
- `conf_main`: Mean $P[i, j]$ over other members of $i$'s `final_cluster`.
- `conf_sub`: Mean $P_{\text{sub}}[i, j]$ over other members of $i$'s `final_cluster`.
- `runnerup_sub`: Other final cluster maximizing mean $P_{\text{sub}}[i, \text{members}]$, with its score.
- Boundary flag: Assigned when $\text{conf}_{\text{sub}} < 0.50$ or $\text{score}_{\text{runnerup}} > \text{conf}_{\text{sub}}$.

In [3]:
# 1. Subsampling 200 runs on random 80% subsets (126 countries)
co_draw = np.zeros((N, N), dtype=np.int32)
co_share = np.zeros((N, N), dtype=np.int32)

for s in range(200):
    rng = np.random.RandomState(seed=s)
    sub_idx = rng.choice(N, size=126, replace=False)
    sub_emb = emb_C_c[sub_idx]
    km = KMeans(n_clusters=7, n_init=10, random_state=s)
    sub_labs = km.fit_predict(sub_emb)
    
    co_draw[np.ix_(sub_idx, sub_idx)] += 1
    co_share[np.ix_(sub_idx, sub_idx)] += (sub_labs[:, None] == sub_labs[None, :])

P_sub = np.zeros((N, N), dtype=np.float64)
valid_pairs = co_draw > 0
P_sub[valid_pairs] = co_share[valid_pairs] / co_draw[valid_pairs]
np.fill_diagonal(P_sub, 1.0)

# 2. Consensus clustering from P_sub
agg_sub = AgglomerativeClustering(n_clusters=7, metric='precomputed', linkage='average')
cons_sub_labels = agg_sub.fit_predict(1.0 - P_sub)
ari_sub_vs_final = adjusted_rand_score(cons_sub_labels, final_cluster)
cons_sub_sizes = pd.Series(cons_sub_labels).value_counts().tolist()

print(f"=== CONSENSUS FROM P_sub ===")
print(f"ARI(cons_P_sub, final_cluster): {ari_sub_vs_final:.4f}")
print(f"Cluster sizes from P_sub:        {cons_sub_sizes}")

# 3. Compute conf_main, conf_sub, runnerup_sub, runnerup_sub_score
conf_main = np.zeros(N)
conf_sub = np.zeros(N)
runnerup_sub = np.zeros(N, dtype=int)
runnerup_sub_score = np.zeros(N)

for i in range(N):
    c_i = final_cluster[i]
    own_members = [j for j in range(N) if final_cluster[j] == c_i and j != i]
    conf_main[i] = np.mean(P[i, own_members])
    conf_sub[i] = np.mean(P_sub[i, own_members])
    
    sub_scores = {}
    for c in range(7):
        if c != c_i:
            other_m = [j for j in range(N) if final_cluster[j] == c]
            sub_scores[c] = np.mean(P_sub[i, other_m])
    best_r = max(sub_scores.keys(), key=lambda c: sub_scores[c])
    runnerup_sub[i] = best_r
    runnerup_sub_score[i] = sub_scores[best_r]

# 4. Boundary flag identification
boundary_flag = (conf_sub < 0.5) | (runnerup_sub_score > conf_sub)

df_flags = pd.DataFrame({
    'Country': order_df['Country'],
    'final_cluster': final_cluster,
    'conf_main': conf_main,
    'conf_sub': conf_sub,
    'runnerup_sub': runnerup_sub,
    'runnerup_score': runnerup_sub_score,
    'boundary_flag': boundary_flag
})

print(f"\n=== BOUNDARY FLAGGED COUNTRIES ({boundary_flag.sum()}) ===")
flagged_view = df_flags[boundary_flag][['Country', 'final_cluster', 'conf_main', 'conf_sub', 'runnerup_sub', 'runnerup_score']]
print(flagged_view.to_string(index=False))

print(f"\n=== MEAN conf_sub PER CLUSTER ===")
for c in range(7):
    mean_val = conf_sub[final_cluster == c].mean()
    print(f"  Cluster {c} (n={(final_cluster == c).sum()}): {mean_val:.4f}")


=== CONSENSUS FROM P_sub ===
ARI(cons_P_sub, final_cluster): 0.8894
Cluster sizes from P_sub:        [29, 29, 25, 23, 19, 19, 14]

=== BOUNDARY FLAGGED COUNTRIES (14) ===
            Country  final_cluster  conf_main  conf_sub  runnerup_sub  runnerup_score
Antigua and Barbuda              4   0.424286  0.391742             1        0.350335
         Cabo Verde              3   0.536818  0.431897             1        0.288613
         Costa Rica              0   0.509138  0.488941             3        0.276254
            Eritrea              1   0.548800  0.459335             5        0.500574
           Ethiopia              0   0.304138  0.247250             4        0.252270
          Indonesia              6   0.551071  0.454104             5        0.329902
            Ireland              0   0.584483  0.492191             4        0.345217
              Japan              6   0.382857  0.309699             0        0.356838
         Madagascar              3   0.371818  0.333841

## 4. Keywords per Final Cluster (c-TF-IDF and Document Contrast)

Prepare masked preamble texts by removing the `[COUNTRY]` placeholder token.
1. **c-TF-IDF**:
   Concatenate all documents in each cluster into a single document. Apply `CountVectorizer(stop_words='english', min_df=3)`.
   $$\text{c-TF-IDF}_{c, t} = \text{tf}_{c, t} \times \log\left(1 + \frac{A}{f_t}\right)$$
   where $A$ is the mean total token count per cluster and $f_t$ is the aggregate term frequency across all clusters.
2. **Document Contrast**:
   Calculate within-cluster document share ($\text{share}_{\text{in}}$) and out-of-cluster document share ($\text{share}_{\text{out}}$).
   Filter for terms with $\text{share}_{\text{in}} \ge 0.30$ and $\frac{\text{share}_{\text{in}}}{\max(\text{share}_{\text{out}}, 0.01)} \ge 2.0$.

In [4]:
# 1. Prepare masked text with [COUNTRY] token removed
def mask_country_name(country, text):
    words = re.findall(r'[a-zA-Z]+', country)
    cap_words = [w for w in words if w and w[0].isupper() and len(w) >= 4]
    terms = set([country] + cap_words)
    sorted_terms = sorted(terms, key=len, reverse=True)
    pattern = r'\b(?:' + '|'.join(re.escape(t) for t in sorted_terms) + r')\b'
    masked_text, count = re.subn(pattern, '[COUNTRY]', text, flags=re.IGNORECASE)
    return masked_text, count

cleaned_texts = []
for idx, r in clusterable_df.iterrows():
    c = r['Country']
    raw_t = ' '.join(str(r['extracted_text']).split())
    m_t, _ = mask_country_name(c, raw_t)
    no_country = re.sub(r'\[COUNTRY\]', ' ', m_t)
    cleaned_texts.append(' '.join(no_country.split()))

clusterable_df['cleaned_text'] = cleaned_texts
clusterable_df['final_cluster'] = final_cluster

# 2. c-TF-IDF Unigrams (top 15)
cluster_docs = []
for c in range(7):
    c_texts = clusterable_df[clusterable_df['final_cluster'] == c]['cleaned_text'].tolist()
    cluster_docs.append(' '.join(c_texts))

cv_uni = CountVectorizer(stop_words='english', min_df=3)
X_uni = cv_uni.fit_transform(cluster_docs).toarray()
vocab_uni = cv_uni.get_feature_names_out()

A_uni = X_uni.sum(axis=1).mean()
f_t_uni = X_uni.sum(axis=0)
c_tfidf_uni = X_uni * np.log(1.0 + A_uni / f_t_uni)

# 3. c-TF-IDF Bigrams (top 10)
cv_bi = CountVectorizer(ngram_range=(2, 2), stop_words='english', min_df=3)
X_bi = cv_bi.fit_transform(cluster_docs).toarray()
vocab_bi = cv_bi.get_feature_names_out()

A_bi = X_bi.sum(axis=1).mean()
f_t_bi = X_bi.sum(axis=0)
c_tfidf_bi = X_bi * np.log(1.0 + A_bi / f_t_bi)

print("=== TOP 15 c-TF-IDF UNIGRAMS AND TOP 10 BIGRAMS PER CLUSTER ===")
c_tfidf_top_unigrams = {}
for c in range(7):
    print(f"\n--- Cluster {c} (n={(final_cluster == c).sum()}) ---")
    top15_u_idx = np.argsort(c_tfidf_uni[c])[::-1][:15]
    top15_u = [(vocab_uni[i], c_tfidf_uni[c, i]) for i in top15_u_idx]
    c_tfidf_top_unigrams[c] = top15_u
    print("Top 15 Unigrams:")
    for rank, (term, score) in enumerate(top15_u, 1):
        print(f"  {rank:>2}. {term:<20}: {score:.4f}")
        
    top10_b_idx = np.argsort(c_tfidf_bi[c])[::-1][:10]
    top10_b = [(vocab_bi[i], c_tfidf_bi[c, i]) for i in top10_b_idx]
    print("Top 10 Bigrams:")
    for rank, (term, score) in enumerate(top10_b, 1):
        print(f"  {rank:>2}. {term:<25}: {score:.4f}")

# 4. Document Contrast (top 12 unigrams)
cv_doc = CountVectorizer(stop_words='english', binary=True)
doc_term = cv_doc.fit_transform(clusterable_df['cleaned_text']).toarray()
vocab_doc = cv_doc.get_feature_names_out()

print("\n=== DOCUMENT CONTRAST UNIGRAMS (share_in >= 0.30, ratio >= 2.0) ===")
contrast_results = {}
for c in range(7):
    in_mask = (final_cluster == c)
    out_mask = ~in_mask
    n_in = in_mask.sum()
    n_out = out_mask.sum()
    
    shares_in = doc_term[in_mask].sum(axis=0) / n_in
    shares_out = doc_term[out_mask].sum(axis=0) / n_out
    ratios = shares_in / np.maximum(shares_out, 0.01)
    
    valid_idx = np.where((shares_in >= 0.30) & (ratios >= 2.0))[0]
    sorted_valid = sorted(valid_idx, key=lambda i: (shares_in[i], ratios[i]), reverse=True)[:12]
    
    print(f"\n--- Cluster {c} Document Contrast ---")
    contrast_results[c] = []
    if len(sorted_valid) == 0:
        print("  No terms met the contrast criteria.")
    for idx in sorted_valid:
        t = vocab_doc[idx]
        contrast_results[c].append((t, shares_in[idx], shares_out[idx], ratios[idx]))
        print(f"  {t:<20}: share_in = {shares_in[idx]:.3f}, share_out = {shares_out[idx]:.3f}, ratio = {ratios[idx]:.2f}")


=== TOP 15 c-TF-IDF UNIGRAMS AND TOP 10 BIGRAMS PER CLUSTER ===

--- Cluster 0 (n=30) ---
Top 15 Unigrams:
   1. constitution        : 101.0336
   2. people              : 87.7649
   3. justice             : 74.7457
   4. god                 : 67.8059
   5. rights              : 60.7856
   6. freedom             : 60.6881
   7. adopt               : 59.1182
   8. nation              : 57.6143
   9. peace               : 56.9450
  10. government          : 56.8327
  11. democratic          : 55.7387
  12. equality            : 55.7119
  13. republic            : 55.3548
  14. society             : 51.3281
  15. human               : 51.0253
Top 10 Bigrams:
   1. constitution people      : 26.0651
   2. almighty god             : 25.5952
   3. adopt enact              : 24.9289
   4. enact constitution       : 21.3676
   5. freedom justice          : 20.3962
   6. inalienable right        : 20.0144
   7. rule law                 : 19.7811
   8. human rights             : 19.2663
   9. ri

## 5. Central Preambles and Boundary Profiles per Final Cluster

For each cluster:
1. Identify the 5 countries closest to the cluster centroid in $C_c$ space by cosine similarity, displaying their first 300 characters.
2. Identify the 5 lowest `conf_sub` countries with their runner-up cluster.
3. Compute summary statistics of `extracted_words` ($n$, median, min, max).

In [5]:
print("=== CENTRAL PREAMBLES (5 CLOSEST TO CENTROID BY COSINE SIMILARITY) ===")
for c in range(7):
    idx_c = np.where(final_cluster == c)[0]
    centroid = emb_C_c[idx_c].mean(axis=0, keepdims=True)
    centroid_norm = centroid / np.linalg.norm(centroid)
    sims = (emb_C_c[idx_c] @ centroid_norm.T).ravel()
    
    top5_local = np.argsort(sims)[::-1][:5]
    top5_global = idx_c[top5_local]
    
    print(f"\n--- Cluster {c} (n={len(idx_c)}) Central Preambles ---")
    for g, l in zip(top5_global, top5_local):
        cntry = order_df.loc[g, 'Country']
        sim_val = sims[l]
        text_snip = str(clusterable_df.loc[g, 'extracted_text'])[:300]
        print(f"  {cntry:<25} (sim: {sim_val:.4f}): '{text_snip}'")

print("\n=== 5 LOWEST conf_sub MEMBERS PER CLUSTER ===")
for c in range(7):
    sub_c = df_flags[df_flags['final_cluster'] == c].sort_values('conf_sub').head(5)
    print(f"\n--- Cluster {c} ---")
    for _, r in sub_c.iterrows():
        print(f"  {r['Country']:<25}: conf_sub = {r['conf_sub']:.4f}, runner-up = Cluster {r['runnerup_sub']} (score: {r['runnerup_score']:.4f})")

print("\n=== WORD COUNT DISTRIBUTIONS PER CLUSTER ===")
wc_table = clusterable_df.groupby('final_cluster')['extracted_words'].agg(
    n='count',
    median='median',
    min='min',
    max='max'
).reset_index()
print(wc_table.to_string(index=False))


=== CENTRAL PREAMBLES (5 CLOSEST TO CENTROID BY COSINE SIMILARITY) ===

--- Cluster 0 (n=30) Central Preambles ---
  Mongolia                  (sim: 0.6981): 'We, the people of Mongolia:consolidating and strengthening the independence and sovereignty of our state,respecting and upholding the human rights and freedom, justice, and unity of our nation, inheriting and cherishing the traditions of our statehood, history and culture, considering and respecting'
  Ghana                     (sim: 0.6818): 'IN THE NAME OF THE ALMIGHTY GOD We the People of Ghana, IN EXERCISE of our natural and inalienable right to establish a framework of government which shall secure for ourselves and posterity the blessings of liberty, equality of opportunity and prosperity; IN A SPIRIT of friendship and peace with al'
  Tajikistan                (sim: 0.6362): 'WE, THE PEOPLE OF TAJIKISTAN, As an inseparable part of the world community, Realizing our debt and our responsibility to the past, present and futur

## 6. Opening Formula Analysis (Form versus Content)

Classify the opening clause of each preamble based on its first 12 words:
- `we_the_people`: Starts with "we" and contains one of `{'people', 'peoples', 'representatives', 'citizens', 'nation', 'nations', 'members', 'deputies', 'delegates'}` within the first 8 words.
- `in_the_name_of`: Starts with "in the name of" (case-insensitive).
- `whereas`: Starts with "whereas" (case-insensitive).
- `other`: All remaining preamble openings.

Examine the contingency table across final clusters, compute Cramer's V, and list opening excerpts for clusters where $>50\%$ of preambles are classified as `other`.

In [6]:
wtp_keywords = {'people', 'peoples', 'representatives', 'citizens', 'nation', 'nations', 'members', 'deputies', 'delegates'}

opening_types = []
for idx, r in clusterable_df.iterrows():
    text = str(r['extracted_text']).strip()
    words = text.split()
    first8_clean = [re.sub(r'[^a-zA-Z]', '', w).lower() for w in words[:8]]
    first_word_clean = re.sub(r'[^a-zA-Z]', '', words[0]).lower() if len(words) > 0 else ''
    
    if text.lower().startswith('in the name of'):
        t = 'in_the_name_of'
    elif first_word_clean == 'whereas':
        t = 'whereas'
    elif first_word_clean == 'we' and any(k in first8_clean for k in wtp_keywords):
        t = 'we_the_people'
    else:
        t = 'other'
    opening_types.append(t)

clusterable_df['opening_type'] = opening_types

# Contingency table: counts and row percentages
ct_opening_counts = pd.crosstab(clusterable_df['final_cluster'], clusterable_df['opening_type'])
ct_opening_pct = pd.crosstab(clusterable_df['final_cluster'], clusterable_df['opening_type'], normalize='index') * 100

print("=== OPENING FORMULA CONTINGENCY TABLE (COUNTS) ===")
print(ct_opening_counts)

print("\n=== OPENING FORMULA CONTINGENCY TABLE (ROW PERCENTAGES %) ===")
print(ct_opening_pct.round(2))

# Association: Cramer's V
chi2_op, p_op, dof_op, _ = chi2_contingency(ct_opening_counts)
v_op = association(ct_opening_counts, method='cramer')
print(f"\nChi2 = {chi2_op:.4f}, p = {p_op:.4e}, Cramer's V = {v_op:.4f}")

# Excerpts for clusters where > 50% are 'other'
print("\n=== PREAMBLES CLASSIFIED AS 'OTHER' (CLUSTERS WITH > 50% OTHER) ===")
for c in range(7):
    row = ct_opening_counts.loc[c]
    total_in_c = row.sum()
    other_cnt = row.get('other', 0)
    pct = other_cnt / total_in_c
    if pct > 0.50:
        print(f"\n--- Cluster {c} ({other_cnt}/{total_in_c} = {pct:.1%} 'other') ---")
        sub_other = clusterable_df[(clusterable_df['final_cluster'] == c) & (clusterable_df['opening_type'] == 'other')]
        for _, r in sub_other.iterrows():
            snip = str(r['extracted_text'])[:60]
            print(f"  {r['Country']:<25}: '{snip}'")


=== OPENING FORMULA CONTINGENCY TABLE (COUNTS) ===
opening_type   in_the_name_of  other  we_the_people  whereas
final_cluster                                               
0                           2      3             22        3
1                           2     12             11        1
2                           1     15              9        0
3                           1     12             10        0
4                           0      5              5       12
5                          10      6              0        1
6                           0      9              5        1

=== OPENING FORMULA CONTINGENCY TABLE (ROW PERCENTAGES %) ===
opening_type   in_the_name_of  other  we_the_people  whereas
final_cluster                                               
0                        6.67  10.00          73.33    10.00
1                        7.69  46.15          42.31     3.85
2                        4.00  60.00          36.00     0.00
3                        4.35  5

## 7. Geographic Region Integration (Natural Earth Cultural Data)

Load country geometries using the exact data source specified in `app.py` (`ne_110m_admin_0_countries.zip`).
Verify the presence of regional and continental metadata (`CONTINENT`), merge on country name, document all unmatched countries, and compute the contingency table and Cramer's V.

In [7]:
# 1. Load Natural Earth geometry as referenced in app.py
ne_zip_path = '/tmp/ne_110m_admin_0_countries.zip'
if not os.path.exists(ne_zip_path):
    world_gdf = gpd.read_file("https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip")
else:
    world_gdf = gpd.read_file(ne_zip_path)

print(f"Loaded Natural Earth dataset with {len(world_gdf)} features.")
print(f"Continent column present: {'CONTINENT' in world_gdf.columns}")

# 2. Merge on Country == NAME (exact match as performed in app.py)
merged_geo = clusterable_df.merge(world_gdf[['NAME', 'CONTINENT']], left_on='Country', right_on='NAME', how='left')

# List unmatched countries
unmatched_countries = merged_geo[merged_geo['CONTINENT'].isna()]['Country'].tolist()
print(f"\nUnmatched countries ({len(unmatched_countries)}):")
print(unmatched_countries)

# 3. Crosstab for matched countries
matched_geo = merged_geo.dropna(subset=['CONTINENT']).copy()
ct_continent = pd.crosstab(matched_geo['final_cluster'], matched_geo['CONTINENT'])

print(f"\n=== CROSSTAB: FINAL CLUSTER BY CONTINENT (MATCHED n={len(matched_geo)}) ===")
print(ct_continent)

# 4. Cramer's V for continent association
chi2_geo, p_geo, dof_geo, _ = chi2_contingency(ct_continent)
v_geo = association(ct_continent, method='cramer')
print(f"\nChi2 = {chi2_geo:.4f}, p = {p_geo:.4e}, Cramer's V = {v_geo:.4f}")

# Attach continent to dataframe (with NaN for unmatched)
clusterable_df['continent'] = merged_geo['CONTINENT'].values


Loaded Natural Earth dataset with 177 features.
Continent column present: True

Unmatched countries (18):
['Andorra', 'Antigua and Barbuda', 'Bahrain', 'Barbados', 'Cabo Verde', 'Comoros', 'Dominica', 'Grenada', 'Kiribati', 'Marshall Islands', 'Micronesia', 'Nauru', 'Saint Kitts and Nevis', 'Saint Lucia', 'Samoa', 'São Tomé and Principe', 'Seychelles', 'Tuvalu']

=== CROSSTAB: FINAL CLUSTER BY CONTINENT (MATCHED n=140) ===
CONTINENT      Africa  Asia  Europe  North America  Oceania  South America
final_cluster                                                             
0                  11     8       3              3        1              1
1                  26     0       0              0        0              0
2                   0     5      19              0        0              0
3                   2     1       3              6        0              9
4                   0     0       2              5        4              0
5                   5    11       0             

## 8. Export Deliverables

Export the two final datasets:
1. `data/processed/cluster_final.csv`: Comprehensive country-level clustering table with refinement, stability, boundary flags, opening types, and continent annotations.
2. `data/processed/cluster_keywords.csv`: Ranked keyword profiles per cluster with c-TF-IDF scores, in-cluster prevalence, and out-of-cluster prevalence.

In [8]:
# 1. Build cluster_final.csv
df_cluster_final = pd.DataFrame({
    'Country': order_df['Country'],
    'status': clusterable_df['status'],
    'extracted_words': clusterable_df['extracted_words'],
    'orig_cluster': cons_df['orig_cluster'],
    'cons_C_c': cons_df['cons_C_c'],
    'final_cluster': final_cluster,
    'conf_main': conf_main,
    'conf_sub': conf_sub,
    'runnerup_sub': runnerup_sub,
    'runnerup_sub_score': runnerup_sub_score,
    'boundary_flag': boundary_flag,
    'opening_type': clusterable_df['opening_type'],
    'continent': clusterable_df['continent']
})

final_csv_path = os.path.join(data_dir, 'cluster_final.csv')
df_cluster_final.to_csv(final_csv_path, index=False)
print(f"Saved {final_csv_path} with shape {df_cluster_final.shape}")

# 2. Build cluster_keywords.csv (Top 15 c-TF-IDF unigrams per cluster with share_in and share_out)
keyword_records = []
for c in range(7):
    in_mask = (final_cluster == c)
    out_mask = ~in_mask
    n_in = in_mask.sum()
    n_out = out_mask.sum()
    
    top15_u = c_tfidf_top_unigrams[c]
    for rank, (term, score) in enumerate(top15_u, 1):
        # Calculate share_in and share_out for this term
        idx_in_doc = np.where(vocab_doc == term)[0]
        if len(idx_in_doc) > 0:
            col_idx = idx_in_doc[0]
            s_in = doc_term[in_mask, col_idx].sum() / n_in
            s_out = doc_term[out_mask, col_idx].sum() / n_out
        else:
            s_in = 0.0
            s_out = 0.0
        keyword_records.append({
            'cluster': c,
            'rank': rank,
            'term': term,
            'c_tfidf': score,
            'share_in': s_in,
            'share_out': s_out
        })

df_keywords = pd.DataFrame(keyword_records)
keywords_csv_path = os.path.join(data_dir, 'cluster_keywords.csv')
df_keywords.to_csv(keywords_csv_path, index=False)
print(f"Saved {keywords_csv_path} with shape {df_keywords.shape}")

# Print first 5 rows of each
print("\n=== FIRST 5 ROWS OF cluster_final.csv ===")
print(df_cluster_final.head(5))

print("\n=== FIRST 5 ROWS OF cluster_keywords.csv ===")
print(df_keywords.head(5))


Saved ../data/processed/cluster_final.csv with shape (158, 13)
Saved ../data/processed/cluster_keywords.csv with shape (105, 6)

=== FIRST 5 ROWS OF cluster_final.csv ===
       Country       status  extracted_words  orig_cluster  cons_C_c  \
0  Afghanistan  preamble_ok              255             2         5   
1      Albania  preamble_ok              114             1         2   
2      Algeria  preamble_ok             1307             2         1   
3      Andorra  preamble_ok              255             5         2   
4       Angola  preamble_ok              580             4         1   

   final_cluster  conf_main  conf_sub  runnerup_sub  runnerup_sub_score  \
0              5   0.699688  0.644122             6            0.403163   
1              2   0.934583  0.881428             5            0.017946   
2              5   0.659687  0.663903             1            0.323976   
3              2   0.718542  0.624729             4            0.180767   
4              1   0.